# F1 · Definición del proyecto y entorno reproducible

**Análisis reproducible del Inventario Público de Glaciares de Chile 2022**  
**Grupo 3 · MCDI500 — Programación para la Ciencia de Datos**

**Integrantes:** Francisco Correa Soto, Gustavo Aloy Angella y Roberto Gaubert Urbina.

## Propósito del notebook
Este cuaderno presenta la problemática, los objetivos y los límites del proyecto; comprueba el entorno de ejecución y realiza un reconocimiento **descriptivo, sin modificaciones**, del archivo original de la Dirección General de Aguas (DGA). La información obtenida servirá para planificar el procesamiento de la Fase 2.

**Delimitación:** aquí no se eliminan registros, imputan valores ni transforman variables. El diagnóstico y las decisiones de procesamiento se implementarán y justificarán en F2.

**Flujo del notebook:** Definir (1–3) → Estructurar y verificar el dato (4) → Configurar el entorno (5) → Reconocer y documentar el inventario (6–7) → Vincular con el mapa (8) → Persistir, validar y versionar (9) → Reflexionar (10). Cada paso es una función y termina en una comprobación con evidencia.

## 1. Configuración documental del proyecto

El diccionario `PROYECTO` centraliza la definición acordada por el equipo: problemática, pregunta, objetivos, alcance, restricciones y supuestos. Mantenerla en una sola estructura evita diferencias entre este notebook, el README y el informe técnico, que usan exactamente la misma formulación. Los datos del inventario provienen de la Dirección General de Aguas (DGA, 2022).

In [1]:
PROYECTO = {
    "titulo": "Análisis reproducible del Inventario Público de Glaciares de Chile 2022",
    "grupo": "Grupo 3",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": ["Francisco Correa Soto", "Gustavo Aloy Angella", "Roberto Gaubert Urbina"],
    "problematica": (
        "Los glaciares son reservas hídricas estratégicas para Chile. La DGA publica el inventario "
        "oficial (26.180 glaciares, 45 variables) y sus totales, pero no un cálculo reproducible de "
        "cuán concentrado está el hielo. Además, la tabla trae tipos mal leídos, columnas vacías y "
        "faltantes encubiertos. Se requiere un procedimiento reproducible que mida esa concentración "
        "y su variación entre tipos de glaciar."
    ),
    "pregunta": (
        "¿Qué tan concentrado está el volumen de hielo estimado en pocos glaciares y regiones, "
        "y cambia esa concentración según la clasificación del glaciar?"
    ),
    "objetivo_general": (
        "Medir, mediante un flujo reproducible en Python, cuán concentrados están el volumen estimado "
        "y la superficie de los glaciares de Chile en pocos glaciares y regiones, y cómo varía esa "
        "concentración según la clasificación del glaciar."
    ),
    "objetivos_especificos": [
        "Verificar la estructura, la calidad y el rol analítico de las 45 variables, y que el volumen se estimó desde el área (F1–F2).",
        "Documentar y ejecutar la limpieza y transformación del inventario sin modificar el archivo original (F2).",
        "Calcular la concentración del volumen y de la superficie: participación del 1 % y del 10 % de glaciares mayores, por región y por clasificación (F3).",
        "Comunicar con tablas y visualizaciones reproducibles si la concentración cambia según la clasificación (F4).",
    ],
    "alcance": {
        "incluye": ["concentración del volumen y de la superficie por glaciar, región y macrozona",
                    "comparación de la concentración entre clasificaciones de glaciar",
                    "equivalente en agua como magnitud complementaria"],
        "excluye": ["tendencias temporales de pérdida de hielo", "modelos predictivos",
                    "cruce con datos climáticos externos",
                    "tratar área y volumen como variables independientes"],
    },
    "restricciones": [
        "El inventario es transversal: el 90,8 % de los registros corresponde a 2019 y el 8,7 % a 2020, sin mediciones repetidas (sección 7.1).",
        "El espesor y el volumen no son mediciones: F_ESP_MED declara el método de Chen y Ohmura (1990), ESP_MED = 28,5·A^0,357. F1 lo registra desde esa columna; la verificación en los datos se hace en F2 (sección 2.3).",
        "La página de descarga de la DGA no declara una licencia (revisada el 11-11-2026): se usa como dato público con fines académicos, citando la fuente con su enlace.",
    ],
    "supuestos": [
        "El inventario oficial es la mejor estimación disponible de cada glaciar.",
        "Las unidades (km², km³, m s.n.m.) son homogéneas entre regiones.",
        "Cada fila representa un glaciar distinto; COD_GLA es su identificador.",
        "Como el volumen es una función del área, ambos se reportan juntos pero no como variables independientes: la concentración del volumen amplifica la del área.",
    ],
}

## 2. Problemática, pregunta y objetivos

El proyecto mide cuán concentrados están el volumen de hielo estimado y la superficie de los glaciares chilenos en pocos glaciares y regiones, y si esa concentración cambia según la clasificación del glaciar. Los objetivos específicos ordenan el trabajo desde la verificación del inventario (F1) hasta la comunicación reproducible de resultados (F4).

La celda siguiente muestra literalmente la formulación acordada en `PROYECTO`, sin introducir interpretaciones nuevas.

In [2]:
from IPython.display import display, Markdown

def mostrar_definicion(proyecto):
    objetivos = "\n".join(f"{i}. {obj}" for i, obj in enumerate(proyecto["objetivos_especificos"], 1))
    display(Markdown(
        f"**Problemática.** {proyecto['problematica']}\n\n"
        f"**Pregunta de investigación.** {proyecto['pregunta']}\n\n"
        f"**Objetivo general.** {proyecto['objetivo_general']}\n\n"
        f"**Objetivos específicos:**\n{objetivos}"
    ))

mostrar_definicion(PROYECTO)

**Problemática.** Los glaciares son reservas hídricas estratégicas para Chile. La DGA publica el inventario oficial (26.180 glaciares, 45 variables) y sus totales, pero no un cálculo reproducible de cuán concentrado está el hielo. Además, la tabla trae tipos mal leídos, columnas vacías y faltantes encubiertos. Se requiere un procedimiento reproducible que mida esa concentración y su variación entre tipos de glaciar.

**Pregunta de investigación.** ¿Qué tan concentrado está el volumen de hielo estimado en pocos glaciares y regiones, y cambia esa concentración según la clasificación del glaciar?

**Objetivo general.** Medir, mediante un flujo reproducible en Python, cuán concentrados están el volumen estimado y la superficie de los glaciares de Chile en pocos glaciares y regiones, y cómo varía esa concentración según la clasificación del glaciar.

**Objetivos específicos:**
1. Verificar la estructura, la calidad y el rol analítico de las 45 variables, y que el volumen se estimó desde el área (F1–F2).
2. Documentar y ejecutar la limpieza y transformación del inventario sin modificar el archivo original (F2).
3. Calcular la concentración del volumen y de la superficie: participación del 1 % y del 10 % de glaciares mayores, por región y por clasificación (F3).
4. Comunicar con tablas y visualizaciones reproducibles si la concentración cambia según la clasificación (F4).

## 3. Alcance, restricciones y supuestos

Se distinguen tres elementos para interpretar correctamente los resultados:

- **Alcance:** decisiones del equipo sobre los análisis incluidos y excluidos.
- **Restricciones:** condiciones impuestas por los datos, el método y el plazo disponible.
- **Supuestos:** condiciones que se adoptan provisionalmente y que deben verificarse cuando sea posible.

La naturaleza transversal del inventario impide inferir tendencias temporales. Asimismo, el volumen y el espesor estimados no deben describirse como mediciones directas. La unicidad de `COD_GLA` se verifica en la sección 7.1; la homogeneidad de las unidades se revisará durante el trabajo.

In [3]:
def mostrar_marco(proyecto):
    for titulo, elementos in (
        ("Incluye", proyecto["alcance"]["incluye"]),
        ("Excluye", proyecto["alcance"]["excluye"]),
        ("Restricciones", proyecto["restricciones"]),
        ("Supuestos", proyecto["supuestos"]),
    ):
        display(Markdown(f"**{titulo}**\n" + "\n".join(f"- {x}" for x in elementos)))

mostrar_marco(PROYECTO)

**Incluye**
- concentración del volumen y de la superficie por glaciar, región y macrozona
- comparación de la concentración entre clasificaciones de glaciar
- equivalente en agua como magnitud complementaria

**Excluye**
- tendencias temporales de pérdida de hielo
- modelos predictivos
- cruce con datos climáticos externos
- tratar área y volumen como variables independientes

**Restricciones**
- El inventario es transversal: el 90,8 % de los registros corresponde a 2019 y el 8,7 % a 2020, sin mediciones repetidas (sección 7.1).
- El espesor y el volumen no son mediciones: F_ESP_MED declara el método de Chen y Ohmura (1990), ESP_MED = 28,5·A^0,357. F1 lo registra desde esa columna; la verificación en los datos se hace en F2 (sección 2.3).
- La página de descarga de la DGA no declara una licencia (revisada el 11-11-2026): se usa como dato público con fines académicos, citando la fuente con su enlace.

**Supuestos**
- El inventario oficial es la mejor estimación disponible de cada glaciar.
- Las unidades (km², km³, m s.n.m.) son homogéneas entre regiones.
- Cada fila representa un glaciar distinto; COD_GLA es su identificador.
- Como el volumen es una función del área, ambos se reportan juntos pero no como variables independientes: la concentración del volumen amplifica la del área.

## 4. Organización del repositorio, módulo compartido y verificación del dataset

La estructura separa los notebooks de cada fase (`F1/`, `F2/`), los datos originales (`data/raw/`), los resultados (`data/processed/`), la documentación (`docs/`) y el **código reutilizable (`src/`)**.

**Módulo `src/utilidades.py`.** Las funciones que usan varias fases (localizar la raíz, calcular y verificar la huella del dataset, cargar el inventario, leer `requirements.txt`, consultar Git, la clase `Bitacora`, las funciones de prueba y `cargar_limpio`, que relee el resultado de F2 con sus tipos) viven en un archivo propio, no copiadas en cada notebook. El módulo define además una excepción del proyecto, `ProyectoError`, que distingue «el dato o el repositorio no son los esperados» de los errores de las librerías. La celda localiza la raíz del repositorio (la carpeta que contiene `src/utilidades.py`), agrega `src/` a la ruta de importación e importa el módulo. Desde ahí todas las rutas son relativas a la raíz.

**Versión del dataset.** Se calcula la huella SHA-256 del CSV y se compara con la registrada cuando el archivo se incorporó al repositorio (commit `ed4dffe`). Una huella distinta indica un archivo modificado o de otra versión, y `verificar_huella` lanza `ProyectoError`.

In [4]:
import sys
import importlib
from pathlib import Path
import numpy as np

# Raíz del repositorio = carpeta que contiene src/utilidades.py (sirve desde F1/ o desde la raíz).
candidatas = [c for c in (Path.cwd(), *Path.cwd().parents) if (c / "src" / "utilidades.py").is_file()]
if not candidatas:
    raise FileNotFoundError("No se encontró src/utilidades.py: abra el notebook desde el repositorio clonado.")
RAIZ = candidatas[0]
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))
import utilidades as utl
importlib.reload(utl)                      # toma la versión actual del módulo si la celda se re-ejecuta

assert utl.localizar_raiz() == RAIZ, "La raíz del módulo y la del dataset no coinciden"
np.random.seed(utl.SEMILLA)

FUENTE = {
    "organismo": "Dirección General de Aguas (DGA), Ministerio de Obras Públicas",
    "conjunto": "Inventario Público de Glaciares de Chile 2022",
    "sitio": "https://dga.mop.gob.cl/inventario-publico-glaciares/",
    "licencia": "no declarada en la página de descarga (revisada el 11-11-2026)",
    "incorporado_al_repositorio": "2026-11-05 (commit ed4dffe)",
}
RUTA_CSV = RAIZ / "data" / "raw" / utl.ARCHIVO_INVENTARIO
huella = utl.verificar_huella(RUTA_CSV)

print("Módulo compartido:", Path(utl.__file__).relative_to(RAIZ))
for clave, valor in FUENTE.items():
    print(f"{clave}: {valor}")
print("Archivo de entrada:", RUTA_CSV.relative_to(RAIZ), f"({RUTA_CSV.stat().st_size / 1024**2:.2f} MB)")
print("SHA-256:", huella)
print("Huella verificada: el CSV es la versión registrada del inventario.")

Módulo compartido: src/utilidades.py
organismo: Dirección General de Aguas (DGA), Ministerio de Obras Públicas
conjunto: Inventario Público de Glaciares de Chile 2022
sitio: https://dga.mop.gob.cl/inventario-publico-glaciares/
licencia: no declarada en la página de descarga (revisada el 11-11-2026)
incorporado_al_repositorio: 2026-11-05 (commit ed4dffe)
Archivo de entrada: data/raw/Inventario_Publico_de_Glaciares_2022.csv (9.46 MB)
SHA-256: 5691de5d7d7ffe2c8586a7066e227ac5cc91fa1ec69f54096e937a054c283f1d
Huella verificada: el CSV es la versión registrada del inventario.


### 4.1 Pruebas del módulo: casos normales, límite y excepciones

Antes de usar el módulo en las fases siguientes se prueba de forma aislada, con archivos temporales cuyo resultado esperado se conoce de antemano. Las excepciones verifican que una entrada inválida produzca **el error correcto**: `FileNotFoundError` si falta un archivo, `ProyectoError` si el dato no es la versión registrada y `ValueError` si se intenta registrar una decisión sin evidencia.

In [5]:
import hashlib
import tempfile
import pandas as pd
from IPython.display import display

from utilidades import espera_error, prueba, ejecutar_pruebas   # funciones de prueba del módulo (también las usa F2)

def bitacora_sin_duplicar():
    b = utl.Bitacora("prueba")
    b.registrar("etapa", "decisión", "1")
    b.registrar("etapa", "decisión", "2")              # misma decisión, evidencia actualizada
    prueba(len(b) == 1 and b.tabla()["evidencia"].iloc[0] == "2")

with tempfile.TemporaryDirectory() as carpeta:
    tmp = Path(carpeta)
    (tmp / "dato.txt").write_bytes(b"glaciar")
    (tmp / "vacio.txt").write_bytes(b"")
    (tmp / "req.txt").write_text("# comentario\npandas==3.1.1\nNumPy == 2.5.4\njupyterlab\n", encoding="utf-8")
    (tmp / "req_sin_versiones.txt").write_text("pandas\nnumpy\n", encoding="utf-8")
    PRUEBAS_MODULO = [
        ("normal", "huella_sha256 coincide con hashlib",
         lambda: prueba(utl.huella_sha256(tmp / "dato.txt") == hashlib.sha256(b"glaciar").hexdigest())),
        ("normal", "verificar_huella acepta el CSV registrado",
         lambda: prueba(utl.verificar_huella(RUTA_CSV) == utl.HUELLA_INVENTARIO)),
        ("normal", "leer_requirements interpreta 'paquete==versión'",
         lambda: prueba(utl.leer_requirements(tmp / "req.txt") == {"pandas": "3.1.1", "numpy": "2.5.4"})),
        ("normal", "email_valido acepta un email institucional",
         lambda: prueba(utl.email_valido("nombre.apellido@uandresbello.edu"))),
        ("límite", "huella de un archivo vacío",
         lambda: prueba(utl.huella_sha256(tmp / "vacio.txt") == hashlib.sha256(b"").hexdigest())),
        ("límite", "localizar_raiz desde la raíz y desde una subcarpeta",
         lambda: prueba(utl.localizar_raiz(RAIZ) == RAIZ == utl.localizar_raiz(RAIZ / "F1"))),
        ("límite", "requirements sin versiones fijas → diccionario vacío",
         lambda: prueba(utl.leer_requirements(tmp / "req_sin_versiones.txt") == {})),
        ("límite", "email de un equipo local (.local) no es válido",
         lambda: prueba(not utl.email_valido("usuario@MacBook-Air.local"))),
        ("límite", "Bitacora no duplica una decisión re-registrada", bitacora_sin_duplicar),
        ("excepción", "localizar_raiz sin data/raw → FileNotFoundError",
         lambda: espera_error(FileNotFoundError, utl.localizar_raiz, tmp)),
        ("excepción", "verificar_huella de otro archivo → ProyectoError",
         lambda: espera_error(utl.ProyectoError, utl.verificar_huella, tmp / "dato.txt")),
        ("excepción", "huella de un archivo inexistente → FileNotFoundError",
         lambda: espera_error(FileNotFoundError, utl.huella_sha256, tmp / "no_existe.txt")),
        ("excepción", "Bitacora.registrar sin evidencia → ValueError",
         lambda: espera_error(ValueError, utl.Bitacora("prueba").registrar, "etapa", "decisión", " ")),
    ]
    resultado_modulo = ejecutar_pruebas(PRUEBAS_MODULO)

display(resultado_modulo)
assert (resultado_modulo["resultado"] == "OK").all(), "Hay pruebas del módulo fallidas"
print(f"Pruebas del módulo: {len(resultado_modulo)}/{len(resultado_modulo)} OK ·",
      resultado_modulo["tipo"].value_counts().to_dict())

,tipo,prueba,resultado
0,normal,huella_sha256 coincide con hashlib,OK
1,normal,verificar_huella acepta el CSV registrado,OK
2,normal,leer_requirements interpreta 'paquete==versión',OK
3,normal,email_valido acepta un email institucional,OK
4,límite,huella de un archivo vacío,OK
5,límite,localizar_raiz desde la raíz y desde una subca...,OK
6,límite,requirements sin versiones fijas → diccionario...,OK
7,límite,email de un equipo local (.local) no es válido,OK
8,límite,Bitacora no duplica una decisión re-registrada,OK
9,excepción,localizar_raiz sin data/raw → FileNotFoundError,OK


Pruebas del módulo: 13/13 OK · {'límite': 5, 'normal': 4, 'excepción': 4}


## 5. Verificación del entorno reproducible

La reproducibilidad exige registrar **qué versiones se usaron realmente**, no solo mencionar las librerías. Cada pieza del ecosistema cumple una función distinta:

| Pieza | Para qué la usamos | Qué falla si se omite |
|---|---|---|
| Entorno virtual (`.venv`) | Aislar las versiones del proyecto | Un paquete «desaparece» o un compañero no puede reproducir |
| JupyterLab + `ipykernel` | Ejecutar el notebook con el intérprete del proyecto | `ModuleNotFoundError` con un paquete que sí está instalado |
| `requirements.txt` (`pip freeze`) | Declarar las versiones exactas, incluidas las dependencias indirectas | «A mí me funciona» y a nadie más |
| `src/utilidades.py` | Escribir una sola vez lo que usan varias fases | La misma función copiada y divergente en cada notebook |
| Git / GitHub | Registrar la historia del trabajo, su autoría y la revisión por Pull Request | No hay evidencia de contribución individual |

La celda siguiente comprueba que el kernel es el del entorno virtual y compara la versión instalada de cada librería con la declarada en `requirements.txt`. Si falta una librería que los notebooks **importan** (NumPy, pandas, scikit-learn, Matplotlib), el notebook se detiene con un mensaje que indica cómo instalarla. Las demás se reportan sin detener la ejecución, y una versión distinta se marca `DIFIERE`. Como verificación cruzada, otro integrante crea el entorno desde `requirements.txt` y ejecuta el notebook completo en su equipo.

In [6]:
import platform
import importlib.metadata as metadata

DEPENDENCIAS = ["numpy", "pandas", "scikit-learn", "matplotlib", "jupyterlab", "ipykernel", "seaborn"]
IMPRESCINDIBLES = ["numpy", "pandas", "scikit-learn", "matplotlib"]   # importadas por F1 y F2

def verificar_entorno(dependencias, ruta_requirements):
    """Compara versiones instaladas con las declaradas; no instala ni modifica nada."""
    declaradas = utl.leer_requirements(ruta_requirements)
    filas = []
    for paquete in dependencias:
        try:
            instalada = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            instalada = None
        declarada = declaradas.get(paquete.lower())
        estado = "NO INSTALADO" if instalada is None else ("OK" if instalada == declarada else "DIFIERE")
        filas.append({"paquete": paquete, "declarada": declarada, "instalada": instalada, "estado": estado})
    return pd.DataFrame(filas)

en_entorno_virtual = sys.prefix != sys.base_prefix
print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("Kernel en entorno virtual:", "sí" if en_entorno_virtual else "NO: seleccionar el kernel de .venv")

tabla_entorno = verificar_entorno(DEPENDENCIAS, RAIZ / "requirements.txt")
display(tabla_entorno)
faltan = tabla_entorno.query("estado == 'NO INSTALADO'")["paquete"].tolist()
assert not set(faltan) & set(IMPRESCINDIBLES), f"Faltan {sorted(set(faltan) & set(IMPRESCINDIBLES))}: ejecutar pip install -r requirements.txt"
print("No instaladas (no imprescindibles):", faltan or "ninguna")
print("Librerías con versión distinta a requirements.txt:", int((tabla_entorno["estado"] == "DIFIERE").sum()))

Python: 3.14.6
Sistema: Linux-7.0.0-43-generic-x86_64-with-glibc2.43
Kernel en entorno virtual: sí


,paquete,declarada,instalada,estado
0,numpy,2.5.4,2.5.4,OK
1,pandas,3.1.1,3.1.1,OK
2,scikit-learn,1.9.1,1.9.1,OK
3,matplotlib,3.11.2,3.11.2,OK
4,jupyterlab,4.6.6,4.6.6,OK
5,ipykernel,7.4.0,7.4.0,OK
6,seaborn,0.13.2,0.13.2,OK


No instaladas (no imprescindibles): ninguna
Librerías con versión distinta a requirements.txt: 0


## 6. Reconocimiento inicial del inventario

En esta etapa se abre el archivo original y se examinan sus **dimensiones, nombres de columnas y tipos inferidos por Pandas**. Es una comprobación de estructura, no un proceso de limpieza.

Los tipos que Pandas asigna al leer un CSV pueden diferir del significado real de la variable: por ejemplo, un código administrativo puede aparecer como entero aunque su función sea categórica. La corrección de tipos y cualquier otra transformación quedan reservadas para F2.

In [7]:
inventario = utl.cargar_inventario(RUTA_CSV)   # lectura sin modificar valores (src/utilidades.py)
print(f"Filas: {inventario.shape[0]:,}")
print(f"Columnas: {inventario.shape[1]}")
display(inventario.head(5))

Filas: 26,180
Columnas: 45


,OBJECTID,Id,COD_GLA,NOMBRE,CLASIFICA,AREA_KM2,REGION,PROVINCIA,COMUNA,DATUM,...,WGI_1,WGI_2,WGI_3,FRENTE_TER,RESOL_IMG,ERROR,OBSERVACIO,CUT_REG,CUT_PROV,CUT_COM
0,174,0,CL101010001@,VOLCAN PARINACOTA,GLACIAR DE MONTAÑA,1.168,ARICA Y PARINACOTA,PARINACOTA,PUTRE,WGS 84,...,6,6,3,TIERRA,15.0,0.11076,NaN,15,152,15201
1,180,0,CL101001023@,VOLCAN POMERAPE,GLACIAR DE MONTAÑA,0.478,ARICA Y PARINACOTA,PARINACOTA,PUTRE,WGS 84,...,6,9,6,TIERRA,15.0,0.08874,NaN,15,152,15201
2,183,0,CL101021004@,S/N,GLACIARETE,0.162,ARICA Y PARINACOTA,PARINACOTA,PUTRE,WGS 84,...,7,7,3,TIERRA,15.0,0.05248,NaN,15,152,15201
3,184,0,CL101021007@,S/N,GLACIARETE,0.094,ARICA Y PARINACOTA,PARINACOTA,PUTRE,WGS 84,...,7,9,6,TIERRA,15.0,0.01984,NaN,15,152,15201
4,185,0,CL101021008@,S/N,GLACIARETE,0.156,ARICA Y PARINACOTA,PARINACOTA,PUTRE,WGS 84,...,7,9,6,TIERRA,15.0,0.02801,NaN,15,152,15201


In [8]:
def describir_estructura(df):
    """Devuelve nombres y tipos inferidos; no modifica el DataFrame."""
    return pd.DataFrame({
        "variable": df.columns,
        "tipo_inferido": [str(tipo) for tipo in df.dtypes],
    })

estructura = describir_estructura(inventario)
display(estructura)

,variable,tipo_inferido
0,OBJECTID,int64
1,Id,int64
2,COD_GLA,str
3,NOMBRE,str
4,CLASIFICA,str
5,AREA_KM2,float64
6,REGION,str
7,PROVINCIA,str
8,COMUNA,str
9,DATUM,str


### 6.1. Roles analíticos previstos

Se identifican las variables que permitirán responder la pregunta del proyecto. `COD_GLA` se considera identificador; región, macrozona, cuenca y clasificación son variables de agrupación; superficie, volumen y equivalente en agua son magnitudes de interés. Esta selección **no implica eliminar las demás columnas**: el inventario completo se mantiene disponible para evaluar su calidad y trazabilidad.

In [9]:
ROLES_ANALITICOS = {
    "identificador": ["COD_GLA"],
    "territorio": ["REGION", "PROVINCIA", "COMUNA", "MZON_GLAC", "NOM_CUEN"],
    "clasificacion": ["CLASIFICA"],
    "superficie": ["AREA_KM2"],
    "volumen_estimado": ["VOL_km3", "EQ_AGUAKM3"],
    "ubicacion": ["LATITUD", "LONGITUD"],
    "altitud": ["HMEDIA", "HMAX", "HMIN"],
    "fecha_inventario": ["INVE_FECHA"],
}

filas = [
    {"rol": rol, "variable": variable, "presente": variable in inventario.columns}
    for rol, variables in ROLES_ANALITICOS.items()
    for variable in variables
]
display(pd.DataFrame(filas))

,rol,variable,presente
0,identificador,COD_GLA,True
1,territorio,REGION,True
2,territorio,PROVINCIA,True
3,territorio,COMUNA,True
4,territorio,MZON_GLAC,True
5,territorio,NOM_CUEN,True
6,clasificacion,CLASIFICA,True
7,superficie,AREA_KM2,True
8,volumen_estimado,VOL_km3,True
9,volumen_estimado,EQ_AGUAKM3,True


### 6.2. Hallazgos preliminares

El inventario contiene **26.180 registros y 45 columnas**. La lectura inicial permite advertir que `LATITUD` y `LONGITUD` se interpretan como texto por el uso de coma decimal; `CUBIERTO` y `OBSERVACIO` aparecen sin datos; y `Id` no aporta variación. Estas señales se registran para orientar el diagnóstico de F2, **sin modificar ninguna celda del archivo**.

La presencia de valores faltantes no basta para justificar la exclusión de una variable. En F2 será necesario distinguir los faltantes estructurales de los problemas de registro y evaluar el impacto de cada decisión.

## 7. Diccionario de las 45 variables y diagnóstico descriptivo inicial

Para documentar la composición del inventario se construye un diccionario que combina una **clasificación temática preliminar** con información observada directamente en el CSV. Se registran nombre, descripción provisional, grupo, rol analítico, tipo leído, cantidad de valores presentes, faltantes y valores únicos.

Las descripciones son una **propuesta de trabajo del equipo**, no equivalen a un diccionario oficial validado por la DGA. En particular, las unidades o significados de `ERROR`, `PERIMETRO`, `PENDIENTE`, `CH_CASQ` y los códigos `WGI_*` requieren contraste documental.

El diagnóstico cuantifica la estructura del archivo, pero no ejecuta conversiones, imputaciones, filtrados ni escalamiento.

In [10]:
# Diccionario documental: no altera el dataset.
DEFINICIONES_VARIABLES = [('OBJECTID', 'Identificación y clasificación', 'Identificador interno de objeto geográfico', 'Identificador técnico'), ('Id', 'Identificación y clasificación', 'Identificador auxiliar', 'Identificador técnico'), ('COD_GLA', 'Identificación y clasificación', 'Código del glaciar', 'Identificador principal'), ('NOMBRE', 'Identificación y clasificación', 'Nombre registrado del glaciar', 'Descriptiva'), ('CLASIFICA', 'Identificación y clasificación', 'Clasificación del glaciar', 'Categórica principal'), ('REGION', 'Territorio e hidrografía', 'Región administrativa', 'Categórica principal'), ('PROVINCIA', 'Territorio e hidrografía', 'Provincia', 'Categórica'), ('COMUNA', 'Territorio e hidrografía', 'Comuna', 'Categórica'), ('NOM_CUEN', 'Territorio e hidrografía', 'Nombre de cuenca', 'Categórica principal'), ('COD_CUEN', 'Territorio e hidrografía', 'Código de cuenca', 'Código categórico'), ('COD_SCUEN', 'Territorio e hidrografía', 'Código de subcuenca', 'Código categórico'), ('COD_SSCUEN', 'Territorio e hidrografía', 'Código de subsubcuenca', 'Código categórico'), ('MZON_GLAC', 'Territorio e hidrografía', 'Macrozona glaciar', 'Categórica principal'), ('CUT_REG', 'Territorio e hidrografía', 'Código administrativo regional', 'Código categórico'), ('CUT_PROV', 'Territorio e hidrografía', 'Código administrativo provincial', 'Código categórico'), ('CUT_COM', 'Territorio e hidrografía', 'Código administrativo comunal', 'Código categórico'), ('DATUM', 'Coordenadas', 'Datum geodésico declarado', 'Metadato espacial'), ('HUSO', 'Coordenadas', 'Huso de referencia declarado', 'Metadato espacial'), ('NORTE', 'Coordenadas', 'Coordenada UTM norte', 'Espacial'), ('ESTE', 'Coordenadas', 'Coordenada UTM este', 'Espacial'), ('LATITUD', 'Coordenadas', 'Latitud geográfica', 'Espacial'), ('LONGITUD', 'Coordenadas', 'Longitud geográfica', 'Espacial'), ('AREA_KM2', 'Propiedades físicas', 'Superficie (km²)', 'Numérica principal'), ('PERIMETRO', 'Propiedades físicas', 'Perímetro; unidad pendiente de confirmar', 'Numérica complementaria'), ('ORIENTA', 'Propiedades físicas', 'Orientación', 'Categórica complementaria'), ('HMEDIA', 'Propiedades físicas', 'Altitud media; unidad por verificar', 'Numérica complementaria'), ('HMAX', 'Propiedades físicas', 'Altitud máxima; unidad por verificar', 'Numérica complementaria'), ('HMIN', 'Propiedades físicas', 'Altitud mínima; unidad por verificar', 'Numérica complementaria'), ('PENDIENTE', 'Propiedades físicas', 'Pendiente; unidad por confirmar', 'Numérica complementaria'), ('ESP_MED', 'Propiedades físicas', 'Espesor medio estimado; unidad por verificar', 'Numérica complementaria'), ('ERROR', 'Propiedades físicas', 'Campo de error; definición por verificar', 'Técnica'), ('F_ESP_MED', 'Estimaciones de hielo y agua', 'Referencia del método de espesor', 'Metadato metodológico'), ('VOL_km3', 'Estimaciones de hielo y agua', 'Volumen estimado (km³)', 'Numérica principal'), ('EQ_AGUAKM3', 'Estimaciones de hielo y agua', 'Equivalente en agua (km³)', 'Numérica principal'), ('CH_CASQ', 'Estimaciones de hielo y agua', 'Código asociado a campo de hielo; significado por verificar', 'Categórica complementaria'), ('FUENTE_DIG', 'Metadatos y clasificación WGI', 'Fuente de digitalización', 'Metadato'), ('FUEN_FECHA', 'Metadatos y clasificación WGI', 'Fecha de la fuente', 'Metadato temporal'), ('INVE_FECHA', 'Metadatos y clasificación WGI', 'Año de inventario', 'Metadato temporal'), ('WGI_1', 'Metadatos y clasificación WGI', 'Código WGI nivel 1; significado por verificar', 'Código categórico'), ('WGI_2', 'Metadatos y clasificación WGI', 'Código WGI nivel 2; significado por verificar', 'Código categórico'), ('WGI_3', 'Metadatos y clasificación WGI', 'Código WGI nivel 3; significado por verificar', 'Código categórico'), ('FRENTE_TER', 'Metadatos y clasificación WGI', 'Tipo de terminación del glaciar', 'Categórica complementaria'), ('RESOL_IMG', 'Metadatos y clasificación WGI', 'Resolución de imagen; unidad por verificar', 'Metadato técnico'), ('CUBIERTO', 'Campos auxiliares', 'Campo cubierto; significado por verificar', 'Por evaluar'), ('OBSERVACIO', 'Campos auxiliares', 'Observaciones', 'Por evaluar')]

def construir_diccionario(df, definiciones):
    """Describe las 45 columnas originales sin modificar sus datos."""
    meta = pd.DataFrame(definiciones, columns=["variable", "grupo", "descripcion_preliminar", "rol_analitico"])
    assert len(meta) == len(df.columns) == 45
    assert set(meta["variable"]) == set(df.columns)
    meta = meta.set_index("variable").loc[df.columns].rename_axis("variable").reset_index()
    meta["tipo_pandas"] = meta["variable"].map(df.dtypes.astype(str).to_dict())
    meta["no_nulos"] = meta["variable"].map(df.notna().sum().to_dict())
    meta["faltantes"] = meta["variable"].map(df.isna().sum().to_dict())
    meta["porcentaje_faltante"] = (100 * meta["faltantes"] / len(df)).round(2)
    meta["valores_unicos"] = meta["variable"].map(df.nunique(dropna=True).to_dict())
    return meta

diccionario_variables = construir_diccionario(inventario, DEFINICIONES_VARIABLES)
display(diccionario_variables)
print("Variables documentadas:", len(diccionario_variables))
print("Grupos temáticos:")
display(diccionario_variables.groupby("grupo").size().rename("variables").to_frame())

,variable,grupo,descripcion_preliminar,rol_analitico,tipo_pandas,no_nulos,faltantes,porcentaje_faltante,valores_unicos
0,OBJECTID,Identificación y clasificación,Identificador interno de objeto geográfico,Identificador técnico,int64,26180,0,0.00,26078
1,Id,Identificación y clasificación,Identificador auxiliar,Identificador técnico,int64,26180,0,0.00,1
2,COD_GLA,Identificación y clasificación,Código del glaciar,Identificador principal,str,26180,0,0.00,26180
3,NOMBRE,Identificación y clasificación,Nombre registrado del glaciar,Descriptiva,str,26180,0,0.00,431
4,CLASIFICA,Identificación y clasificación,Clasificación del glaciar,Categórica principal,str,26180,0,0.00,5
5,AREA_KM2,Propiedades físicas,Superficie (km²),Numérica principal,float64,26180,0,0.00,2323
6,REGION,Territorio e hidrografía,Región administrativa,Categórica principal,str,26180,0,0.00,16
7,PROVINCIA,Territorio e hidrografía,Provincia,Categórica,str,26180,0,0.00,40
8,COMUNA,Territorio e hidrografía,Comuna,Categórica,str,26180,0,0.00,84
9,DATUM,Coordenadas,Datum geodésico declarado,Metadato espacial,str,26180,0,0.00,1


Variables documentadas: 45
Grupos temáticos:


,variables
grupo,
Campos auxiliares,2
Coordenadas,6
Estimaciones de hielo y agua,4
Identificación y clasificación,5
Metadatos y clasificación WGI,8
Propiedades físicas,9
Territorio e hidrografía,11


### 7.1. Señales de calidad y decisiones pendientes

Se resumen indicadores que justifican una revisión posterior: campos completamente vacíos, variables constantes, identificadores que deben comprobarse y columnas con tipos incompatibles con su interpretación prevista.

**Criterio metodológico:** primero se presenta la evidencia; después, en F2, se comparan alternativas y se decide si conviene conservar, corregir o excluir una variable. No se anticipan decisiones irreversibles basadas únicamente en el porcentaje de valores faltantes.

In [11]:
def diagnostico_documental(df, diccionario):
    """Resume señales de calidad; no modifica ni devuelve datos transformados."""
    campos = ["OBJECTID", "Id", "COD_GLA", "CUBIERTO", "OBSERVACIO", "CH_CASQ", "LATITUD", "LONGITUD", "FUEN_FECHA", "F_ESP_MED"]
    return diccionario.set_index("variable").loc[campos, ["tipo_pandas", "faltantes", "porcentaje_faltante", "valores_unicos"]]

display(diagnostico_documental(inventario, diccionario_variables))
print("Filas duplicadas completas:", int(inventario.duplicated().sum()))
print("COD_GLA duplicados (sin contar nulos):", int(inventario.loc[inventario["COD_GLA"].notna(), "COD_GLA"].duplicated().sum()))
print("Valores de ejemplo de LATITUD (sin conversión):", inventario["LATITUD"].dropna().head(3).tolist())
print("Ceros en HMIN / VOL_km3 / EQ_AGUAKM3:", int((inventario["HMIN"] == 0).sum()), "/",
      int((inventario["VOL_km3"] == 0).sum()), "/", int((inventario["EQ_AGUAKM3"] == 0).sum()))
print("Año de inventario (INVE_FECHA, % de registros):",
      (100 * inventario["INVE_FECHA"].value_counts(normalize=True)).round(1).to_dict())
print("Categorías de FUENTE_DIG (posibles variantes de escritura):", sorted(inventario["FUENTE_DIG"].dropna().unique()))

DECISIONES_F2 = pd.DataFrame([
    ("LATITUD / LONGITUD", "Revisar coma decimal y rangos", "Evaluar conversión numérica sin alterar archivo raw"),
    ("FUEN_FECHA", "Revisar formatos de fecha", "Evaluar conversión a fecha"),
    ("CUBIERTO / OBSERVACIO", "Verificar ausencia total de datos", "Evaluar exclusión justificada"),
    ("Id / OBJECTID", "Revisar constantes y repetición", "No usarlos como identificador principal sin verificar"),
    ("CH_CASQ", "Investigar faltantes y definición", "No eliminar automáticamente"),
    ("WGI_1 / WGI_2 / WGI_3", "Confirmar significado de códigos", "Tratar como categorías cuando corresponda"),
    ("HMIN / VOL_km3 / EQ_AGUAKM3", "Ceros que podrían ser faltantes encubiertos", "Evaluar si son valores reales o ausencia de dato"),
    ("FUENTE_DIG", "Misma categoría escrita de distintas formas", "Evaluar normalización del texto"),
], columns=["variables", "hallazgo_a_investigar", "decision_pendiente_F2"])
display(DECISIONES_F2)

,tipo_pandas,faltantes,porcentaje_faltante,valores_unicos
variable,,,,
OBJECTID,int64,0,0.00,26078
Id,int64,0,0.00,1
COD_GLA,str,0,0.00,26180
CUBIERTO,float64,26180,100.00,0
OBSERVACIO,float64,26180,100.00,0
CH_CASQ,str,25457,97.24,6
LATITUD,str,0,0.00,26180
LONGITUD,str,0,0.00,26180
FUEN_FECHA,str,0,0.00,264


Filas duplicadas completas: 0
COD_GLA duplicados (sin contar nulos): 0
Valores de ejemplo de LATITUD (sin conversión): ['-18,1744803305003', '-18,1245109467846', '-18,4280225838322']
Ceros en HMIN / VOL_km3 / EQ_AGUAKM3: 69 / 362 / 668
Año de inventario (INVE_FECHA, % de registros): {2019: 90.8, 2020: 8.7, 2021: 0.4, 2023: 0.0, 2022: 0.0}
Categorías de FUENTE_DIG (posibles variantes de escritura): ['EIA ANDINA 244 IMG:RAPIDEYE', 'FOTOGRAMETR├ìA', 'GE01', 'GOOGLE EARTH', 'GPR Y DRON', 'LANDSAT OLI', 'LIDAR', 'LiDAR ANDINA', 'PLANETSCOPE', 'PLEIADES', 'RAPIDEYE', 'SAF GEOTEC 1:50.000', 'SENTINEL 2', 'SPOT 6', 'SPOT-5 GOOGLE EARTH', 'VUELO HYCON', 'WV02', 'WV03', 'WV04']


,variables,hallazgo_a_investigar,decision_pendiente_F2
0,LATITUD / LONGITUD,Revisar coma decimal y rangos,Evaluar conversión numérica sin alterar archiv...
1,FUEN_FECHA,Revisar formatos de fecha,Evaluar conversión a fecha
2,CUBIERTO / OBSERVACIO,Verificar ausencia total de datos,Evaluar exclusión justificada
3,Id / OBJECTID,Revisar constantes y repetición,No usarlos como identificador principal sin ve...
4,CH_CASQ,Investigar faltantes y definición,No eliminar automáticamente
5,WGI_1 / WGI_2 / WGI_3,Confirmar significado de códigos,Tratar como categorías cuando corresponda
6,HMIN / VOL_km3 / EQ_AGUAKM3,Ceros que podrían ser faltantes encubiertos,Evaluar si son valores reales o ausencia de dato
7,FUENTE_DIG,Misma categoría escrita de distintas formas,Evaluar normalización del texto


### 7.2 Ficha del conjunto y evaluación contra los criterios del curso

El **rol analítico**, no el tipo de dato, determina el preprocesamiento de F2. Por eso cada variable se clasifica en los roles del curso: identificador, continua, discreta, binaria, nominal, ordinal, fecha y alta cardinalidad, además de *vacía* y *constante*, que no cuentan como roles útiles. Varias columnas `int64` son en realidad **códigos** (`COD_CUEN`, `CUT_COM`), `INVE_FECHA` es un año discreto y `LATITUD`/`LONGITUD`, leídas como texto, son continuas.

La función `evaluar_criterios` aplica a la ficha los **mismos umbrales que `validar_dataset.py`**, la herramienta del curso: los requisitos *obligatorios* descalifican el conjunto si no se cumplen, y los *avisos* no lo descalifican, pero lo que falte se declara como restricción.

**Referencia en APA 7:** Dirección General de Aguas. (2022). *Inventario público de glaciares de Chile 2022* [Conjunto de datos]. Ministerio de Obras Públicas. https://dga.mop.gob.cl/inventario-publico-glaciares/

In [12]:
from datetime import date

ROL_CURSO = {
    "OBJECTID": "identificador", "Id": "constante", "COD_GLA": "identificador", "NOMBRE": "alta cardinalidad",
    "CLASIFICA": "nominal", "AREA_KM2": "continua", "REGION": "nominal", "PROVINCIA": "alta cardinalidad",
    "COMUNA": "alta cardinalidad", "DATUM": "constante", "HUSO": "constante", "NORTE": "continua",
    "ESTE": "continua", "FUENTE_DIG": "alta cardinalidad", "FUEN_FECHA": "fecha", "INVE_FECHA": "discreta",
    "NOM_CUEN": "alta cardinalidad", "COD_CUEN": "alta cardinalidad", "COD_SCUEN": "alta cardinalidad",
    "COD_SSCUEN": "alta cardinalidad", "MZON_GLAC": "ordinal", "CH_CASQ": "nominal", "CUBIERTO": "vacía",
    "LATITUD": "continua", "LONGITUD": "continua", "PERIMETRO": "continua", "ORIENTA": "nominal",
    "HMEDIA": "continua", "HMAX": "continua", "HMIN": "continua", "PENDIENTE": "continua",
    "ESP_MED": "continua", "F_ESP_MED": "constante", "VOL_km3": "continua", "EQ_AGUAKM3": "continua",
    "WGI_1": "nominal", "WGI_2": "nominal", "WGI_3": "nominal", "FRENTE_TER": "nominal",
    "RESOL_IMG": "discreta", "ERROR": "continua", "OBSERVACIO": "vacía", "CUT_REG": "nominal",
    "CUT_PROV": "alta cardinalidad", "CUT_COM": "alta cardinalidad",
}
assert set(ROL_CURSO) == set(inventario.columns), "Cada columna debe tener un rol declarado"
diccionario_variables["rol_curso"] = diccionario_variables["variable"].map(ROL_CURSO)

FICHA = {
    "titulo": "Inventario Público de Glaciares de Chile 2022",
    "organismo": "Dirección General de Aguas (DGA)",
    "url": "https://dga.mop.gob.cl/inventario-publico-glaciares/",
    "anio_publicacion": 2022,
    "fuente_verificable": True,
    "licencia_explicita": False,          # la página de descarga no declara licencia (revisada el 11-11-2026)
    "unidad_observacion": "Un glaciar",
    "filas": inventario.shape[0],
    "columnas": inventario.shape[1],
    "tamano_mb": round(RUTA_CSV.stat().st_size / 1024**2, 2),
    "duplicadas": int(inventario.duplicated().sum()),
    "pct_faltante_max": float(diccionario_variables["porcentaje_faltante"].max()),
    "sobre_60": diccionario_variables.query("porcentaje_faltante > 60")["variable"].tolist(),
    "roles": diccionario_variables["rol_curso"].value_counts().to_dict(),
}

def evaluar_criterios(ficha, anio_actual):
    """Aplica a la ficha los umbrales de validar_dataset.py más dos requisitos de la Semana 1."""
    r = ficha["roles"]
    utiles = [rol for rol in r if rol not in ("identificador", "vacía", "constante")]
    numericas = r.get("continua", 0) + r.get("discreta", 0)
    categoricas = r.get("nominal", 0) + r.get("binaria", 0) + r.get("ordinal", 0)
    requisitos = [  # (obligatorio, requisito, cumple, detalle, por qué se pide)
        (True, "Al menos 2000 filas", ficha["filas"] >= 2000, f"{ficha['filas']:,} filas", "Agrupar por categoría sin grupos vacíos"),
        (True, "Al menos 12 columnas", ficha["columnas"] >= 12, f"{ficha['columnas']} columnas", "Variedad de roles analíticos"),
        (True, "Combina al menos 3 roles analíticos", len(utiles) >= 3, ", ".join(sorted(utiles)), "El conjunto debe ser multivariado"),
        (True, "Al menos una variable numérica", numericas >= 1, f"{numericas} (continuas o discretas)", "Escalamiento y atípicos"),
        (True, "Al menos una variable categórica", categoricas >= 1, f"{categoricas} (nominales, binarias u ordinales)", "Codificación de categorías"),
        (False, "Presencia de valores faltantes (≥ 1 %)", ficha["pct_faltante_max"] >= 1, f"máximo {ficha['pct_faltante_max']} %", "Sin faltantes no hay preprocesamiento que justificar"),
        (False, "Ninguna variable sobre 60 % de faltantes", not ficha["sobre_60"], ", ".join(ficha["sobre_60"]) or "ninguna", "Por encima, la variable no es recuperable"),
        (False, "Incluye alguna variable de fecha", r.get("fecha", 0) >= 1, f"{r.get('fecha', 0)} detectada(s)", "Parseo y derivación de variables temporales"),
        (False, "Incluye texto o alta cardinalidad", r.get("alta cardinalidad", 0) >= 1, f"{r.get('alta cardinalidad', 0)} detectadas", "Normalización y agrupación"),
        (False, "Archivo bajo 100 MB", ficha["tamano_mb"] <= 100, f"{ficha['tamano_mb']} MB", "Límite de GitHub"),
        (False, "Sin filas duplicadas exactas", ficha["duplicadas"] == 0, f"{ficha['duplicadas']} duplicadas", "Cada fila, una observación"),
        (False, "Licencia abierta y fuente verificable", ficha["fuente_verificable"] and ficha["licencia_explicita"],
         "fuente oficial verificable; licencia no declarada", "Uso legítimo y procedencia comprobable"),
        (False, "Datos de los últimos cinco años", ficha["anio_publicacion"] >= anio_actual - 5,
         f"publicado en {ficha['anio_publicacion']}", "Problema contingente"),
    ]
    return pd.DataFrame([{"estado": "OK" if ok else ("FALTA" if oblig else "AVISO"), "tipo": "obligatorio" if oblig else "aviso",
                          "requisito": req, "detalle": det, "por_que_se_pide": porque}
                         for oblig, req, ok, det, porque in requisitos])

evaluacion_dataset = evaluar_criterios(FICHA, date.today().year)
display(diccionario_variables["rol_curso"].value_counts().rename("variables").to_frame())
display(evaluacion_dataset)
assert "FALTA" not in evaluacion_dataset["estado"].values, "El conjunto no cumple un requisito obligatorio"
avisos = evaluacion_dataset.query("estado == 'AVISO'")["requisito"].tolist()
print(f"Obligatorios: todos cumplidos · avisos ({len(avisos)}):", avisos)

,variables
rol_curso,
continua,14
alta cardinalidad,10
nominal,9
constante,4
identificador,2
discreta,2
vacía,2
fecha,1
ordinal,1


,estado,tipo,requisito,detalle,por_que_se_pide
0,OK,obligatorio,Al menos 2000 filas,"26,180 filas",Agrupar por categoría sin grupos vacíos
1,OK,obligatorio,Al menos 12 columnas,45 columnas,Variedad de roles analíticos
2,OK,obligatorio,Combina al menos 3 roles analíticos,"alta cardinalidad, continua, discreta, fecha, ...",El conjunto debe ser multivariado
3,OK,obligatorio,Al menos una variable numérica,16 (continuas o discretas),Escalamiento y atípicos
4,OK,obligatorio,Al menos una variable categórica,"10 (nominales, binarias u ordinales)",Codificación de categorías
5,OK,aviso,Presencia de valores faltantes (≥ 1 %),máximo 100.0 %,Sin faltantes no hay preprocesamiento que just...
6,AVISO,aviso,Ninguna variable sobre 60 % de faltantes,"CH_CASQ, CUBIERTO, OBSERVACIO","Por encima, la variable no es recuperable"
7,OK,aviso,Incluye alguna variable de fecha,1 detectada(s),Parseo y derivación de variables temporales
8,OK,aviso,Incluye texto o alta cardinalidad,10 detectadas,Normalización y agrupación
9,OK,aviso,Archivo bajo 100 MB,9.46 MB,Límite de GitHub


Obligatorios: todos cumplidos · avisos (2): ['Ninguna variable sobre 60 % de faltantes', 'Licencia abierta y fuente verificable']


**Lectura.** El inventario cumple todos los requisitos obligatorios y deja **dos avisos**, que no lo descalifican y se declaran como restricciones:

1. **Variables sobre 60 % de faltantes** (`CUBIERTO`, `OBSERVACIO` y `CH_CASQ`). Siguiendo el criterio del curso, F2 las descarta, pero conserva el contenido de `CH_CASQ` como bandera binaria (*pertenece o no a un campo de hielo*).
2. **Licencia no declarada.** La fuente es oficial y verificable, pero la página de descarga de la DGA no declara una licencia (revisada el 11-11-2026). El conjunto se usa como dato público con fines académicos, citando la fuente, y se registra como restricción.

El conjunto **no tiene variables binarias originales**. El requisito «binaria u ordinal» lo cubre `MZON_GLAC` (ordinal), y F2 deriva variables binarias (`EN_CAMPO_HIELO`, `FRENTE_AGUA`).

## 8. Correspondencia con las fases y con el mapa conceptual

| Fase | Propósito | Evidencia esperada |
|---|---|---|
| **F1 — Definición** | Delimitar la pregunta, documentar el entorno y reconocer el inventario | Este notebook, `src/utilidades.py` y `docs/` |
| **F2 — Preparación** | Diagnosticar, limpiar, transformar y validar | `F2/F2_Procesamiento.ipynb`, pruebas y `data/processed/` |
| **F3 — Análisis** | Medir la concentración del volumen y la superficie por glaciar, región y clasificación | Tablas y curvas de concentración reproducibles |
| **F4 — Comunicación** | Comunicar si la concentración cambia según la clasificación | Figuras, interpretación y productos finales |

La tabla siguiente vincula cada elemento del mapa conceptual (`docs/f1/mapa_conceptual_fase1.drawio`) con su implementación: tipo de elemento, estado y evidencia. Se construye como datos, se verifica que todo lo implementado tenga evidencia y se exporta a `docs/vinculacion_mapa_conceptual.csv`.

In [13]:
VINCULACION = [  # (elemento del mapa v2, tipo, estado, evidencia)
    ("A · Situación, problema y aporte frente a la DGA", "Proceso", "Implementado", "Secciones 1–3: PROYECTO['problematica']; README"),
    ("A · Pregunta acotada: concentración del volumen y su variación por clasificación", "Proceso", "Implementado", "Sección 2: PROYECTO['pregunta']"),
    ("A · Dataset DGA 2022 con licencia revisada (no declarada por la DGA)", "Flujo reproducible", "Implementado", "Sección 7.2: FICHA y evaluación del dataset (aviso de licencia)"),
    ("A · Supuesto verificado: el volumen se estimó desde el área", "Decisión técnica", "Implementado", "F2 sección 2.3: ESP_MED = 28,5·A^0,357; sección 1 supuestos"),
    ("B① Definición", "Proceso", "Implementado", "Secciones 1–3"),
    ("B② Obtención: CSV en data/raw verificado con SHA-256", "Flujo reproducible", "Implementado", "Sección 4: verificar_huella"),
    ("B③ Exploración: faltantes encubiertos, atípicos y categorías", "Proceso", "Implementado", "Secciones 6–7 (descriptiva); F2 sección 2"),
    ("B④ Transformación → data/processed", "Proceso", "Implementado en F2", "F2 secciones 3–8; data/processed/"),
    ("B⑤ Análisis de concentración · B⑥ Comunicación", "Proceso", "Proyectado (F3–F4)", ""),
    ("C · Python + .venv + requirements.txt → ① configura", "Herramienta", "Implementado", "Sección 5: kernel y versiones verificadas"),
    ("C · src/utilidades.py → ② verifica y carga", "Herramienta", "Implementado", "Sección 4: módulo importado y probado (13 casos)"),
    ("C · pandas · NumPy + validar_dataset.py → ③ mide", "Herramienta", "Implementado", "Secciones 6–7; F2 sección 2; docs/validacion_dataset.md"),
    ("C · pandas · scikit-learn → ④ transforma", "Herramienta", "Implementado en F2", "F2 secciones 3–5"),
    ("C · pandas (groupby) → ⑤ agrega", "Herramienta", "Proyectado (F3)", ""),
    ("C · Matplotlib · seaborn → ⑥ visualiza", "Herramienta", "Parcial: gráficos de control en F2; figuras en F4", "F2 secciones 3.5, 4 y 5"),
    ("C · Git + GitHub y JupyterLab (transversales)", "Herramienta", "Implementado", "Historial de Git; Pull Requests integrados a main; sección 9.1"),
    ("D · Estructura: F1/, F2/, data/, src/, docs/ en la raíz", "Organización del repositorio", "Implementado", "Sección 9: validar_fase1"),
    ("D · results/ (tablas y figuras)", "Organización del repositorio", "Proyectado (F4)", ""),
    ("D · Protocolo: notebooks ejecutados, rama + PR, commits por integrante", "Decisión técnica", "Implementado con ajuste: desarrollo a cargo de un integrante, con autorización docente", "Historial de Git y README"),
    ("E · Notebook: Markdown + código + salidas", "Estructura del notebook", "Implementado", "Narrativa antes de cada bloque en F1 y F2"),
    ("E · Bitácora de decisiones con cifras", "Documentación", "Implementado", "docs/bitacora_f2.md"),
    ("E · Informe + referencias APA 7 (DGA con enlace)", "Documentación", "Implementado", "docs/informe/; sección 7.2"),
]
vinculacion = pd.DataFrame(VINCULACION, columns=["elemento_mapa", "tipo", "estado", "evidencia"])
implementados = vinculacion["estado"].str.startswith(("Implementado", "Parcial"))
assert vinculacion.loc[implementados, "evidencia"].str.strip().ne("").all(), "Hay elementos implementados sin evidencia"
tipos_requeridos = {"Herramienta", "Proceso", "Flujo reproducible", "Decisión técnica", "Organización del repositorio", "Estructura del notebook"}
assert tipos_requeridos <= set(vinculacion["tipo"]), "Faltan tipos de elemento exigidos por la rúbrica"
with pd.option_context("display.max_colwidth", None):
    display(vinculacion)
print(vinculacion["estado"].value_counts().to_dict())

,elemento_mapa,tipo,estado,evidencia
0,"A · Situación, problema y aporte frente a la DGA",Proceso,Implementado,Secciones 1–3: PROYECTO['problematica']; README
1,A · Pregunta acotada: concentración del volumen y su variación por clasificación,Proceso,Implementado,Sección 2: PROYECTO['pregunta']
2,A · Dataset DGA 2022 con licencia revisada (no declarada por la DGA),Flujo reproducible,Implementado,Sección 7.2: FICHA y evaluación del dataset (aviso de licencia)
3,A · Supuesto verificado: el volumen se estimó desde el área,Decisión técnica,Implementado,"F2 sección 2.3: ESP_MED = 28,5·A^0,357; sección 1 supuestos"
4,B① Definición,Proceso,Implementado,Secciones 1–3
5,B② Obtención: CSV en data/raw verificado con SHA-256,Flujo reproducible,Implementado,Sección 4: verificar_huella
6,"B③ Exploración: faltantes encubiertos, atípicos y categorías",Proceso,Implementado,Secciones 6–7 (descriptiva); F2 sección 2
7,B④ Transformación → data/processed,Proceso,Implementado en F2,F2 secciones 3–8; data/processed/
8,B⑤ Análisis de concentración · B⑥ Comunicación,Proceso,Proyectado (F3–F4),
9,C · Python + .venv + requirements.txt → ① configura,Herramienta,Implementado,Sección 5: kernel y versiones verificadas


{'Implementado': 15, 'Implementado en F2': 2, 'Proyectado (F3–F4)': 1, 'Proyectado (F3)': 1, 'Parcial: gráficos de control en F2; figuras en F4': 1, 'Proyectado (F4)': 1, 'Implementado con ajuste: desarrollo a cargo de un integrante, con autorización docente': 1}


**Justificación.** F1 materializa lo que el mapa asigna a la definición (①), la obtención inmutable (②) y las herramientas que intervienen en esas etapas. El mapa v2 incorpora la retroalimentación docente: la pregunta se acotó a la concentración del volumen; el supuesto de que el volumen se estimó desde el área quedó verificado en los datos (F2 sección 2.3); la licencia quedó revisada y su ausencia, declarada como restricción; y cada herramienta aparece conectada con la etapa en que interviene. La exploración (③) se inicia aquí solo de forma descriptiva y se completa en F2, para no invadir esa fase. El análisis de concentración (⑤) y la comunicación (⑥) siguen siendo pertinentes y parten de `data/processed/glaciares_limpio.csv`;

## 9. Persistencia y validación de la fase

La fase deja sus artefactos documentales en `docs/`, generados desde las estructuras de este notebook para que no puedan contradecirse con él:

- `diccionario_variables.csv` (45 variables con su rol);
- `evaluacion_criterios_dataset.csv`;
- `vinculacion_mapa_conceptual.csv`;
- `metadatos_fase1.json` (condiciones de ejecución).

Luego, `verificar_f1` comprueba el dataset y `validar_fase1` comprueba con `assert` lo que la fase debe dejar:

1. la estructura de carpetas;
2. los artefactos del entorno;
3. la definición completa del problema;
4. la coherencia del diccionario con el CSV;
5. el módulo importable;
6. los artefactos de `docs/`.

**No se corrige ningún dato.** Finalmente, la sección 9.1 registra la rama y el commit con que se generaron estas salidas.

In [14]:
import json

DOCS = RAIZ / "docs"
diccionario_variables.to_csv(DOCS / "diccionario_variables.csv", index=False)
evaluacion_dataset.to_csv(DOCS / "evaluacion_criterios_dataset.csv", index=False)
vinculacion.to_csv(DOCS / "vinculacion_mapa_conceptual.csv", index=False)

interprete = Path(sys.executable)
METADATOS_F1 = {
    "proyecto": PROYECTO["titulo"],
    "grupo": PROYECTO["grupo"],
    "fase": "F1",
    "fecha_ejecucion": date.today().isoformat(),
    "semilla": utl.SEMILLA,
    "python": sys.version.split()[0],
    "sistema": f"{platform.system()} {platform.release()}",
    "interprete": interprete.relative_to(RAIZ).as_posix() if interprete.is_relative_to(RAIZ) else interprete.name,
    "entorno_virtual": en_entorno_virtual,
    "dependencias": dict(zip(tabla_entorno["paquete"], tabla_entorno["instalada"])),
    "dataset": {"archivo": utl.ARCHIVO_INVENTARIO, "organismo": FICHA["organismo"], "url": FICHA["url"],
                "filas": FICHA["filas"], "columnas": FICHA["columnas"], "sha256": huella},
}
(DOCS / "metadatos_fase1.json").write_text(json.dumps(METADATOS_F1, indent=2, ensure_ascii=False), encoding="utf-8")

ARTEFACTOS_DOCS = ["diccionario_variables.csv", "evaluacion_criterios_dataset.csv",
                   "vinculacion_mapa_conceptual.csv", "metadatos_fase1.json"]
for nombre in ARTEFACTOS_DOCS:
    print(f"docs/{nombre:36} {(DOCS / nombre).stat().st_size:6} bytes")

docs/diccionario_variables.csv              5239 bytes
docs/evaluacion_criterios_dataset.csv       1377 bytes
docs/vinculacion_mapa_conceptual.csv        2676 bytes
docs/metadatos_fase1.json                    832 bytes


In [15]:
def verificar_f1(df):
    """Comprueba las condiciones mínimas del dataset para iniciar F2; no modifica el DataFrame."""
    requeridas = {"COD_GLA", "REGION", "CLASIFICA", "AREA_KM2", "VOL_km3", "EQ_AGUAKM3"}
    assert not df.empty, "El archivo no contiene registros"
    assert requeridas.issubset(df.columns), "Faltan columnas clave"
    assert df.shape == (26180, 45), "Revisar la versión del inventario: se esperaban 26.180 × 45"
    assert df["COD_GLA"].notna().all() and df["COD_GLA"].is_unique, "COD_GLA debe ser único y sin nulos"
    print("Verificación F1 superada: lectura, estructura e identificador disponibles.")

def validar_fase1(raiz, config, diccionario, df):
    """Comprueba lo que la Fase 1 debe dejar en el repositorio; lanza AssertionError si algo falta."""
    esperadas = ["data/raw", "data/processed", "docs", "src", "F1", "F2"]
    faltan = [c for c in esperadas if not (raiz / c).is_dir()]
    assert not faltan, f"Faltan carpetas: {faltan}"
    print(f"[OK] Estructura: {', '.join(esperadas)}")

    ignorado = (raiz / ".gitignore").read_text(encoding="utf-8")
    for regla in (".venv/", ".ipynb_checkpoints/", "__pycache__/"):
        assert regla in ignorado, f".gitignore no excluye {regla}"
    for nombre in ("requirements.txt", "README.md"):
        assert (raiz / nombre).stat().st_size > 0, f"{nombre} está vacío"
    print("[OK] Artefactos del entorno: .gitignore (excluye .venv, cachés y checkpoints), requirements.txt, README.md")

    for clave in ("titulo", "problematica", "pregunta", "objetivo_general", "objetivos_especificos", "alcance", "restricciones", "supuestos"):
        assert config.get(clave), f"La definición no declara '{clave}'"
    assert len(config["objetivos_especificos"]) >= 3, "Se esperan al menos tres objetivos específicos"
    print(f"[OK] Definición completa: {len(config['objetivos_especificos'])} objetivos específicos, alcance, restricciones y supuestos")

    assert list(diccionario["variable"]) == list(df.columns) and diccionario["rol_curso"].notna().all()
    print(f"[OK] Diccionario coherente con el CSV: {len(diccionario)} variables con rol declarado")

    assert Path(utl.__file__).resolve() == (raiz / "src" / "utilidades.py").resolve()
    print("[OK] Módulo src/utilidades.py importado desde el repositorio")

    for nombre in ARTEFACTOS_DOCS:
        assert (raiz / "docs" / nombre).stat().st_size > 0, f"docs/{nombre} vacío"
    print(f"[OK] Artefactos de docs/: {len(ARTEFACTOS_DOCS)} archivos")

verificar_f1(inventario)
validar_fase1(RAIZ, PROYECTO, diccionario_variables, inventario)
print("F1 finalizada: no se modificó ni exportó el inventario.")

Verificación F1 superada: lectura, estructura e identificador disponibles.
[OK] Estructura: data/raw, data/processed, docs, src, F1, F2
[OK] Artefactos del entorno: .gitignore (excluye .venv, cachés y checkpoints), requirements.txt, README.md
[OK] Definición completa: 4 objetivos específicos, alcance, restricciones y supuestos
[OK] Diccionario coherente con el CSV: 45 variables con rol declarado
[OK] Módulo src/utilidades.py importado desde el repositorio
[OK] Artefactos de docs/: 4 archivos
F1 finalizada: no se modificó ni exportó el inventario.


### 9.1 Trazabilidad con el repositorio

Se registran la rama y el commit sobre los que se ejecutó este notebook, para vincular sus salidas con el historial de Git. Se excluyen del control de cambios los artefactos de `docs/` que el propio notebook regenera en cada ejecución. Si Git no está disponible, la función informa «no disponible» en lugar de fallar.

In [16]:
SALIDAS_F1 = [f"docs/{nombre}" for nombre in ARTEFACTOS_DOCS]   # artefactos que F1 regenera
for clave, valor in utl.version_repositorio(RAIZ, excluir=SALIDAS_F1).items():
    print(f"{clave}: {valor}")

rama: main
commit: 02671f4
cambios_sin_commit: False


## 10. Reflexión técnica

### Hallazgos
- **El conjunto cumple los requisitos obligatorios, pero no es «perfecto», y eso es una ventaja.** La evaluación de 7.2 deja dos avisos: tres variables sobre 60 % de faltantes y una licencia no declarada. Detectarlos aquí orientó decisiones de F2, como descartar esas columnas y recuperar `CH_CASQ` como variable binaria.
- **El rol analítico, y no el tipo de dato, define el trabajo pendiente.** Hay códigos leídos como enteros (`COD_CUEN`, `CUT_COM`), un año discreto (`INVE_FECHA`) y coordenadas continuas leídas como texto (`LATITUD`, `LONGITUD`). Cada una exige en F2 un tratamiento distinto.
- **No todos los faltantes son nulos.** El diagnóstico descriptivo (7.1) encontró 69 `HMIN = 0`, 362 `VOL_km3 = 0` y 668 `EQ_AGUAKM3 = 0` en columnas que el perfil reporta con 0 % de faltantes. F2 los investiga y los trata como faltantes encubiertos o redondeos.

## Referencias y cierre de fase

**Fuente de datos.** Dirección General de Aguas. (2022). *Inventario público de glaciares de Chile 2022* [Conjunto de datos]. Ministerio de Obras Públicas. https://dga.mop.gob.cl/inventario-publico-glaciares/

**Referencias técnicas.** McKinney, W. (2022). *Python for data analysis* (3.ª ed.). O’Reilly Media · Kluyver, T., et al. (2016). Jupyter Notebooks: A publishing format for reproducible computational workflows. IOS Press. https://doi.org/10.3233/978-1-61499-649-1-87 · Chen, J., & Ohmura, A. (1990). Estimation of Alpine glacier water resources and their change since the 1870s. *IAHS Publication, 193*, 127–135.

**Trazabilidad.** El historial de Git y los Pull Requests registran cada cambio de este notebook, y la sección 9.1 identifica el commit de esta ejecución. El CSV de `data/raw/` permanece intacto, como lo demuestra la huella SHA-256 de la sección 4.

**Cierre de fase.** F1 documenta el problema, el entorno, el módulo compartido y el reconocimiento inicial del inventario. La evaluación detallada de la calidad y toda intervención sobre los datos corresponden a F2.